# 01: Explore regimes and unusual moves

Run the pipeline first (`python -m marketpulse.cli all --demo` or the real-data steps). Set `DATASET` below.

Write your observations in the markdown cells. The notebook is only useful once it holds your own conclusions.

In [ ]:
import sys, logging
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))
logging.getLogger("hmmlearn").setLevel(logging.ERROR)

import pandas as pd
import matplotlib.pyplot as plt
from marketpulse.config import load_config
from marketpulse.pipeline import Store
from marketpulse.regimes import regime_names

DATASET = "demo"   # "real" after you download data
cfg = load_config()
store = Store(cfg, DATASET)
levels, changes = store.load("levels"), store.load("changes")
proba = store.load("regime_proba")
names = regime_names(proba.filter(like="p_").shape[1])
levels.tail()

## 1. Daily changes
Fat tails show up as kurtosis well above 3. Which markets have the fattest tails?

In [ ]:
summary = pd.DataFrame({"std": changes.std(), "skew": changes.skew(), "kurtosis": changes.kurt() + 3})
summary.round(2)

## 2. Regimes over time (out-of-sample, filtered)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
spx = levels["SPX"].loc[proba.index]
for i, n in enumerate(names):
    m = proba["regime"] == i
    axes[0].scatter(spx.index[m], spx[m], s=2, label=n)
axes[0].set_yscale("log"); axes[0].legend(); axes[0].set_title("S&P 500 by regime")
proba.filter(like="p_").set_axis(names, axis=1).plot.area(ax=axes[1], linewidth=0)
axes[1].set_title("Filtered regime probabilities")
plt.tight_layout()

**Observation:** [Do known stress periods land in the stressed regime? How many days late?]

In [ ]:
pd.read_csv(store.reports / "regime_profile.csv", index_col=0).round(2)

In [ ]:
pd.read_csv(store.reports / "regime_transitions.csv", index_col=0).round(3)

## 3. Does volatility really differ by regime?
If the daily std of each market barely changes across regimes, regime-conditional z-scores add nothing over a rolling baseline.

In [ ]:
joined = changes.loc[proba.index].assign(regime=proba["regime"].map(dict(enumerate(names))))
joined.groupby("regime").std().round(2)

## 4. Flags per method

In [ ]:
flags = {k: store.load(f"flags_{k}") for k in ("regime", "rolling", "fixed")}
pd.DataFrame({k: v.sum() for k, v in flags.items()})

In [ ]:
flagged = store.load("flagged_moves")
flagged.sort_values("z", key=abs, ascending=False).head(15)

**Observation:** [Which flags look like real events, and which look like noise?]